In [35]:
import torch.nn as nn
import torch
import pandas as pd
import json
import numpy as np
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import precision_score, recall_score, f1_score

In [181]:
class LSTMAutoencoder(nn.Module):
    def __init__(
        self,
        input_dim: int = 1,
        hidden_dim: int = 64,
        latent_dim: int = 20,
        num_layers: int = 1,
    ):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers

        # ---------- Encoder ---------- #
        self.encoder = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
        )

        # ---------- Bottleneck ---------- #
        self.to_latent = nn.Linear(hidden_dim, latent_dim)
        self.from_latent = nn.Linear(latent_dim, hidden_dim)

        # ---------- Decoder ---------- #
        self.decoder = nn.LSTM(
            input_size=hidden_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
        )

        # ---------- Output ---------- #
        self.output_layer = nn.Linear(hidden_dim, input_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if x.ndim != 3:
            raise ValueError(f"Expected (B,T,F), got {x.shape}")

        batch_size, seq_len, _ = x.shape
        _, (h, _) = self.encoder(x)
        h_last = h[-1]

        latent = self.to_latent(h_last)
        hidden_dec = self.from_latent(latent)

        dec_in = hidden_dec.unsqueeze(1).repeat(1, seq_len, 1)

        h0 = hidden_dec.unsqueeze(0).repeat(self.num_layers, 1, 1)
        c0 = torch.zeros_like(h0)

        out, _ = self.decoder(dec_in, (h0, c0))
        out = self.output_layer(out)

        return out

In [182]:
def load_series(csv_path: str) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df = df.sort_values("timestamp").reset_index(drop=True)
    return df


def load_windows(json_path: str, file_key: str) -> list:
    with open(json_path, "r") as f:
        data = json.load(f)
    return data[file_key]


def build_labels(df: pd.DataFrame, windows: list) -> pd.DataFrame:
    labels = np.zeros(len(df))

    for start, end in windows:
        start = pd.to_datetime(start)
        end = pd.to_datetime(end)

        mask = (df["timestamp"] >= start) & (df["timestamp"] <= end)
        labels[mask] = 1

    df["label"] = labels
    return df

In [183]:
def make_windows_train(values: np.ndarray, labels: np.ndarray, seq_len: int) -> tuple[np.ndarray, np.ndarray]:
    values = np.asarray(values).reshape(-1, 1)
    labels = np.asarray(labels)

    n_windows = len(values) - seq_len + 1
    if n_windows <= 0:
        raise ValueError("Sequence length is greater than the total series length.")

    x_windows = np.empty((n_windows, seq_len, 1), dtype=np.float32)
    y_windows = np.empty((n_windows,), dtype=np.float32)

    for i in range(n_windows):
        x_windows[i] = values[i : i + seq_len]
        y_windows[i] = np.max(labels[i : i + seq_len])

    return x_windows, y_windows


def train_model(
    model: torch.nn.Module,
    train_x: np.ndarray,
    lr: float = 1e-3,
    epochs: int = 10,
    batch_size: int = 64,
    device: str = "cpu",
) -> torch.nn.Module:
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    # ---- Section Name ----
    # Initialize learning rate scheduler to decay LR when validation loss plateaus
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, 
        mode="min", 
        factor=0.5, 
        patience=3
    )
    
    criterion = torch.nn.MSELoss()
    model.train()
    
    x = torch.tensor(train_x, dtype=torch.float32)
    if x.ndim != 3:
        raise ValueError(f"Invalid input shape: {x.shape}")

    dataset = TensorDataset(x)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, pin_memory=(device == "cuda"))

    for epoch in range(epochs):
        total_loss = 0.0
        n_batches = 0

        for (batch,) in loader:
            batch = batch.to(device)
            pred = model(batch)

            loss = criterion(pred, batch)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            n_batches += 1

        epoch_loss = total_loss / max(n_batches, 1)
        
        # ---- Section Name ----
        # Update lr scheduler and extract current learning rate for tracking
        scheduler.step(epoch_loss)
        current_lr = optimizer.param_groups[0]["lr"]

        print(f"epoch {epoch+1}/{epochs} | loss {epoch_loss:.6f} | lr {current_lr:.6f}")

    return model

In [184]:
def make_windows_inference(values: np.ndarray, seq_len: int) -> np.ndarray:
    values = np.asarray(values).reshape(-1, 1)
    n_windows = len(values) - seq_len + 1
    if n_windows <= 0:
        raise ValueError("Sequence length is greater than the total series length.")

    x_windows = []
    for i in range(n_windows):
        x_windows.append(values[i : i + seq_len])

    return np.stack(x_windows).astype(np.float32)


def run_inference(
    model: torch.nn.Module, values: np.ndarray, seq_len: int, device: str = "cpu", batch_size: int = 256
) -> tuple[np.ndarray, np.ndarray]:
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    model.eval()

    if values.ndim == 3:
        x = values
    else:
        x = make_windows_inference(values, seq_len)

    x_tensor = torch.tensor(x, dtype=torch.float32)
    dataset = TensorDataset(x_tensor)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

    errors = []
    recons = []

    with torch.no_grad():
        for (batch,) in loader:
            batch = batch.to(device)
            recon = model(batch)

            err = torch.mean((recon - batch) ** 2, dim=(1, 2))
            errors.append(err.cpu())
            recons.append(recon.cpu())

    return torch.cat(errors).numpy(), torch.cat(recons).numpy()


def compute_threshold(train_errors: np.ndarray, percentile: float = 95.0) -> float:
    return float(np.percentile(train_errors, percentile))


def detect_anomalies(errors: np.ndarray, threshold: float) -> np.ndarray:
    return (np.asarray(errors) > threshold).astype(np.int32)


def inference_pipeline(
    model: torch.nn.Module, train_values: np.ndarray, test_values: np.ndarray, seq_len: int
) -> tuple[np.ndarray, np.ndarray, float, np.ndarray]:
    train_err, _ = run_inference(model, train_values, seq_len)
    threshold = compute_threshold(train_err, 95.0)
    test_err, _ = run_inference(model, test_values, seq_len)
    preds = detect_anomalies(test_err, threshold)

    return train_err, test_err, threshold, preds

In [185]:
device = "cuda" if torch.cuda.is_available() else "cpu"

SEQ_LEN = 150

input_dim = 1
hidden_dim = 128
latent_dim = 4
num_layers = 1

epochs = 100
batch_size = 64
lr = 0.001

In [ ]:
print("device:", device)

# ---- load data ---- #
df = load_series("../assets/data/art_daily_flatmiddle.csv")
windows = load_windows(
    json_path="../assets/labels/combined_windows.json",
    file_key="artificialWithAnomaly/art_daily_flatmiddle.csv",
)
df = build_labels(df, windows)

values = df["value"].values.astype(np.float32)
labels = df["label"].values.astype(np.float32)

# ---- windowing ---- #
x, y = make_windows_train(values, labels, SEQ_LEN) # type: ignore

# ---- TRUE ANOMALY STATS ---- #
true_anomaly_ratio = np.mean(y == 1)
print(f"true_anomaly_ratio: {true_anomaly_ratio:.6f} ({true_anomaly_ratio*100:.2f}%)")
print(f"windows: {len(y)}")
print(f"anomaly_windows: {np.sum(y == 1)}")
print(f"normal_windows: {np.sum(y == 0)}")

# ---- NORMAL ONLY TRAIN SET ---- #
train_x = x[y == 0]

# ---- model ---- #
model = LSTMAutoencoder(input_dim=input_dim, hidden_dim=hidden_dim, latent_dim=latent_dim, num_layers=num_layers)

# ---- train ---- #
model = train_model(model=model, train_x=train_x, epochs=epochs, batch_size=batch_size, device=device, lr=lr)

# ---- save model ---- #
torch.save(model.state_dict(), "lstm_ae_model.pt")
print("model saved")

device: cuda
true_anomaly_ratio: 0.142158 (14.22%)
windows: 3883
anomaly_windows: 552
normal_windows: 3331
epoch 1/100 | loss 2332.628929 | lr 0.001000
epoch 2/100 | loss 2195.572669 | lr 0.001000
epoch 3/100 | loss 2109.925143 | lr 0.001000
epoch 4/100 | loss 2137.985706 | lr 0.001000


In [ ]:
# ---- config ---- #
print("device:", device)

# ---- load data ---- #
df = load_series("../assets/data/art_daily_flatmiddle.csv")
windows = load_windows(
    json_path="../assets/labels/combined_windows.json",
    file_key="artificialWithAnomaly/art_daily_flatmiddle.csv",
)
df = build_labels(df, windows)

values = df["value"].values.astype(np.float32)
labels = df["label"].values.astype(np.float32)

# ---- windowing (GROUND TRUTH SPACE) ---- #
x, y = make_windows_train(values, labels, SEQ_LEN)  # type: ignore

# ---- stats ---- #
true_anomaly_ratio = np.mean(y == 1)
print(f"true_anomaly_ratio: {true_anomaly_ratio:.6f} ({true_anomaly_ratio*100:.2f}%)")
print(f"windows: {len(y)}")
print(f"anomaly_windows: {np.sum(y == 1)}")
print(f"normal_windows: {np.sum(y == 0)}")

# ---- load model ---- #
model = LSTMAutoencoder(input_dim=input_dim, hidden_dim=hidden_dim, latent_dim=latent_dim, num_layers=num_layers)
model.load_state_dict(torch.load("lstm_ae_model.pt", map_location=device))
model.to(device)
model.eval()

# ---- TRAIN distribution errors ---- #
train_x = x[y == 0]
train_err, _ = run_inference(model, train_x, SEQ_LEN, device=device)

# ---- TEST (FULL SERIES) ---- #
test_err, _ = run_inference(model, values, SEQ_LEN, device=device)  # type: ignore

print("\n" + "=" * 80)
print(f"{'Percentile':<12}{'Threshold':<12}{'Anomalies':<12}{'Precision':<12}{'Recall':<12}{'F1-Score':<12}")
print("=" * 80)

# ---- Section Name ----
# Grid sweep execution over the requested percentile scale from 10% to 100%
percentiles = np.arange(10, 101, 1)

for p in percentiles:
    current_threshold = float(np.percentile(train_err, p))
    preds = (test_err > current_threshold).astype(np.int32)
    
    precision = precision_score(y, preds, zero_division=0)
    recall = recall_score(y, preds, zero_division=0)
    f1 = f1_score(y, preds, zero_division=0)
    
    print(f"{p:<12}{current_threshold:<12.4f}{preds.sum():<12}{precision:<12.4f}{recall:<12.4f}{f1:<12.4f}")

print("=" * 80)

device: cuda
true_anomaly_ratio: 0.142158 (14.22%)
windows: 3883
anomaly_windows: 552
normal_windows: 3331

Percentile  Threshold   Anomalies   Precision   Recall      F1-Score    
10          12.2754     3523        0.1493      0.9529      0.2582      
11          12.9813     3486        0.1497      0.9457      0.2585      
12          14.0880     3443        0.1487      0.9275      0.2563      
13          14.6652     3402        0.1481      0.9130      0.2549      
14          15.3004     3366        0.1491      0.9094      0.2563      
15          16.0243     3332        0.1504      0.9076      0.2580      
16          16.4646     3299        0.1519      0.9076      0.2602      
17          16.9207     3263        0.1529      0.9040      0.2616      
18          17.1851     3228        0.1540      0.9004      0.2630      
19          17.7103     3192        0.1548      0.8949      0.2639      
20          18.1430     3157        0.1562      0.8931      0.2658      
21          18.5

In [173]:
from sklearn.ensemble import IsolationForest

# ---- 1. Preprocess for Isolation Forest (Flatten the sequences) ---- #
n_samples, seq_len, n_features = x.shape
x_flatten = x.reshape(n_samples, seq_len * n_features)

# ---- 2. Train Isolation Forest ---- #
# contamination is the expected ratio of anomalies in the data (true_anomaly_ratio)
contamination_ratio = float(np.mean(y == 1))

iso_forest = IsolationForest(
    contamination=contamination_ratio,
    random_state=42,
    n_jobs=-1
)

# IF takes full data or normal only; traditional unsupervised IF fits on the test/full distribution
iso_forest.fit(x_flatten)

# ---- 3. Predict Anomaly Scores ---- #
# Isolation Forest returns 1 for normal, -1 for anomaly
if_preds = iso_forest.predict(x_flatten)

# Map to our label space: 1 for anomaly, 0 for normal
if_preds_mapped = np.where(if_preds == -1, 1, 0)

# ---- 4. Evaluate Baseline ---- #
if_precision = precision_score(y, if_preds_mapped, zero_division=0)
if_recall = recall_score(y, if_preds_mapped, zero_division=0)
if_f1 = f1_score(y, if_preds_mapped, zero_division=0)

# ---- output ---- #
print("=" * 30)
print("Isolation Forest Baseline Results:")
print("-" * 30)
print(f"Precision: {if_precision:.4f}")
print(f"Recall:    {if_recall:.4f}")
print(f"F1-Score:  {if_f1:.4f}")
print("=" * 30)

Isolation Forest Baseline Results:
------------------------------
Precision: 0.4149
Recall:    0.4149
F1-Score:  0.4149
